# Concept Transformer assessment walkthrough

We will build this notebook step by step from the technical assessment and the supplied ICLR 2022 paper. It currently covers **Step 2: Environment Setup**, **Step 3: Data Download & Preparation**, and a **Step 4 debug training/evaluation run**. Bounding-box cropping and an original-code debug training/evaluation run are complete. A regular batch-size-32 training run is now in progress; full results are pending.

The selected environment is a project-local uv `.venv` using **Python 3.8.20 (x86_64 under Rosetta)** and the repository's original model dependency versions. PyTorch runs on **CPU** on this Mac.

## 2. Environment Setup

This section records the repository revision, Python environment, package versions, compatibility decisions, and a PyTorch device check. The package manager is **uv** and the environment is **`.venv`**.

**Architecture choice:** although this Mac has Apple Silicon, we agreed to use Rosetta for this reproduction. SciPy 1.7.1 and pandas 1.3.3 have no macOS ARM64 wheels; their Intel wheels avoid needing to compile the old scientific stack. The original PyTorch 1.10.0 predates Apple MPS support, so this Mac will run the original implementation on **CPU**. CUDA GPU training remains possible on an appropriate NVIDIA system.

### 2.1 Project paths and Python interpreter

Start Jupyter from this project directory. Edit `PROJECT_DIR` if you launch it elsewhere. All project paths are derived from this variable.

In [1]:
from pathlib import Path
import importlib.metadata as metadata
import os
import json
import platform
import subprocess
import sys
try:
    import tomllib
except ImportError:
    import tomli as tomllib

PROJECT_DIR = Path.cwd().resolve()  # Configure this if Jupyter starts elsewhere.
REPO_DIR = PROJECT_DIR / "concept_transformer"
# Loader root: the parent of the inner CUB_200_2011 directory.
DATA_DIR = PROJECT_DIR / "CUB-200-2011" / "CUB_200_2011"
EXPECTED_ENV = PROJECT_DIR / ".venv"

assert (REPO_DIR / "cvit_cub.py").is_file(), "Set PROJECT_DIR to the assessment directory."
assert Path(sys.prefix).resolve() == EXPECTED_ENV.resolve(), (
    "Select this project's .venv kernel, then restart and run again."
)
assert sys.version_info[:2] == (3, 8), "Select the project Python 3.8 kernel."

# Vendored source provenance stays valid after cloning this assessment repository.
repo_commit = json.loads((PROJECT_DIR / "upstream.json").read_text())["commit"]
print("Project:", PROJECT_DIR)
print("Repository commit:", repo_commit)
print("Python executable:", sys.executable)
print("Python version:", platform.python_version())
print("Python architecture:", platform.machine())
print("Operating system:", platform.platform())

Project: /Users/mengyuanz/Documents/projects/concept_transformer_explain
Repository commit: 84c9c30108a6249e017d580c4120ea54af0064f7
Python executable: /Users/mengyuanz/Documents/projects/concept_transformer_explain/.venv/bin/python
Python version: 3.8.20
Python architecture: x86_64
Operating system: macOS-26.6.2-x86_64-i386-64bit


### 2.2 Install and launch

The environment uses the uv-managed **Intel Python 3.8.20** interpreter. To recreate it on this Mac, run from the project directory:

```bash
uv python install cpython-3.8.20-macos-x86_64-none
uv venv --python cpython-3.8.20-macos-x86_64-none .venv
uv sync --locked --python .venv/bin/python
uv run --locked python -m ipykernel install --sys-prefix \
  --name concept_transformer --display-name "Concept Transformer (Python 3.8 Rosetta)"
uv run --locked jupyter lab concept_transformer_walkthrough.ipynb
```

Select **Concept Transformer (Python 3.8 Rosetta)** as the notebook kernel. `uv.lock` records the resolved direct and transitive package versions. On this Mac, `platform.machine()` should report `x86_64` inside this kernel. The interpreter check prevents accidentally running the notebook using Python 3.12.

Installation runs outside the notebook so that packages are not replaced in a running kernel. After changing dependencies, restart the kernel before running all cells.

### 2.3 Dependency versions

The table prints the original repository requirement, our selected version, and what is actually installed in the current kernel. Model dependencies use the versions in `concept_transformer/requirements.txt`. Additional pins cover notebook tooling and transitive compatibility. `uv.lock` records the complete resolved environment.

In [2]:
def read_pins(lines):
    pins = {}
    for line in lines:
        requirement = line.split("#", 1)[0].strip()
        if not requirement:
            continue
        name, separator, version = requirement.partition("==")
        pins[name.lower().replace("_", "-")] = version if separator else "unpinned"
    return pins

original = read_pins((REPO_DIR / "requirements.txt").read_text().splitlines())
manifest = tomllib.loads((PROJECT_DIR / "pyproject.toml").read_text())
selected = read_pins(manifest["project"]["dependencies"])

print(f"{'Package':27} {'Upstream':12} {'Selected':12} {'Installed':12}")
print("-" * 67)
missing = []
mismatched = []
for name, requested_version in selected.items():
    try:
        installed_version = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed_version = "MISSING"
        missing.append(name)
    if installed_version != "MISSING" and installed_version != requested_version:
        mismatched.append(name)
    print(f"{name:27} {original.get(name, 'additional'):12} {requested_version:12} {installed_version:12}")
print("\nuv.lock present:", (PROJECT_DIR / "uv.lock").is_file())
print("Missing packages:", ", ".join(missing) or "none")
print("Version differences:", ", ".join(mismatched) or "none")

Package                     Upstream     Selected     Installed   
-------------------------------------------------------------------
torch                       1.10.0       1.10.0       1.10.0      
torchvision                 0.11.1       0.11.1       0.11.1      
pytorch-lightning           1.4.8        1.4.8        1.4.8       
lightning-bolts             0.3.4        0.3.4        0.3.4       
torchmetrics                0.5          0.5.0        0.5.0       
scipy                       1.7.1        1.7.1        1.7.1       
numpy                       1.22.0       1.22.0       1.22.0      
pandas                      1.3.3        1.3.3        1.3.3       
setuptools                  59.5.0       59.5.0       59.5.0      
albumentations              1.0.3        1.0.3        1.0.3       
timm                        0.4.12       0.4.12       0.4.12      
matplotlib                  3.5.1        3.5.1        3.5.1       
epyc                        unpinned     1.7.2        1.7.2  

### 2.4 Compatibility log

| Finding | Resolution |
| --- | --- |
| **NumPy documentation mismatch:** the GitHub README specifies **1.20.3**, while `requirements.txt` specifies **1.22.0**. | Use **NumPy 1.22.0** because the README's installation command delegates to `requirements.txt`. This discrepancy is documented explicitly; it is not an installer error. |
| The original package pins are incompatible with the initial Python 3.12 environment. | Switch to **Python 3.8.20** and restore every explicitly pinned model dependency from `requirements.txt`. |
| SciPy 1.7.1 and pandas 1.3.3 have no macOS ARM64 wheels. Native builds would require an additional compiler/toolchain. | Use **Intel (`x86_64`) Python 3.8 under Rosetta**, as agreed. This permits the original wheel versions to install without changing those pins. |
| Original PyTorch 1.10.0 predates the Apple MPS backend. | Use **CPU** on this Mac. Rosetta is an intentional compatibility choice for reproducing the older stack; this setup does not use the Apple GPU. |
| The earlier proposed Lightning 1.9.5 setup would have required trainer API changes. | Restore **Lightning 1.4.8**, Bolts 0.3.4, and TorchMetrics 0.5. The repository's original trainer arguments can be retained. |
| The repository leaves `epyc` unpinned and does not pin the complete transitive dependency tree. | Select `epyc==1.7.2`; add compatible transitive and notebook-tool pins in `pyproject.toml`. Lock the complete resolution in `uv.lock`. |
| The first notebook check reported `IProgress not found` from tqdm. | Add `ipywidgets==8.1.5` so notebook progress bars are available. |
| Initial sandbox attempts could not reach PyPI. | Network-enabled download/install commands were subsequently approved. Keep the earlier failure logs and the Python 3.8 installation log in `setup_logs/`. |

References: [GitHub README](https://github.com/IBM/concept_transformer/blob/main/README.md), [GitHub requirements.txt](https://github.com/IBM/concept_transformer/blob/main/requirements.txt), [SciPy 1.7.1 wheels](https://pypi.org/project/scipy/1.7.1/), and [pandas 1.3.3 wheels](https://pypi.org/project/pandas/1.3.3/).

### 2.5 Check package consistency and repository imports

This checks dependency metadata, imports the repository's training dependencies, and requests CLI help without downloading data or pretrained weights. Passing it does not validate a CUB training run.

In [3]:
assert not missing and not mismatched, "Run uv sync, restart the kernel, and rerun from the top."

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], capture_output=True, text=True
)
print(check.stdout.strip())
if check.stderr:
    print(check.stderr.strip())
check.check_returncode()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd
import scipy
import matplotlib
import torch
import torchvision
import pytorch_lightning as pl
import torchmetrics
import timm
import albumentations
import epyc
from pl_bolts.optimizers.lr_scheduler import LinearWarmupCosineAnnealingLR
from ctc import CTCModel, cub_cvit
from data import CUB2011Parts

help_result = subprocess.run(
    [sys.executable, "cvit_cub.py", "--help"], cwd=REPO_DIR,
    capture_output=True, text=True
)
if help_result.returncode:
    print(help_result.stdout)
    print(help_result.stderr)
help_result.check_returncode()
print("Repository imports and cvit_cub.py --help passed.")

No broken requirements found.


Repository imports and cvit_cub.py --help passed.


### 2.6 Verify PyTorch and identify the compute device

`cuda` is the NVIDIA GPU backend, `mps` is the Apple GPU backend, and `cpu` is the fallback. First we test CPU computation; if a GPU backend is available, we test it too. A successful gradient check verifies basic tensor computation and automatic differentiation on that device.

In [4]:
cuda_available = torch.cuda.is_available()
mps_backend = getattr(torch.backends, "mps", None)
mps_built = mps_backend is not None and mps_backend.is_built()
mps_available = mps_backend is not None and mps_backend.is_available()

DEVICE = torch.device("cuda" if cuda_available else "mps" if mps_available else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA available:", cuda_available)
print("MPS built:", mps_built)
print("MPS available:", mps_available)
print("Device for this check:", DEVICE)
if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))

check_devices = [torch.device("cpu")]
if DEVICE.type != "cpu":
    check_devices.append(DEVICE)
for check_device in check_devices:
    x = torch.arange(4, dtype=torch.float32, device=check_device, requires_grad=True)
    loss = x.square().sum()
    loss.backward()
    assert loss.item() == 14.0
    torch.testing.assert_close(x.grad, 2 * x.detach())
    print(f"{check_device}: forward and backward computation passed.")

print("The original trainer uses CPU on this Mac; its GPU selection supports CUDA.")

PyTorch: 1.10.0
CUDA available: False
MPS built: False
MPS available: False
Device for this check: cpu
cpu: forward and backward computation passed.
The original trainer uses CPU on this Mac; its GPU selection supports CUDA.


### 2.7 Save the verified environment record

Run this only after the preceding checks succeed. It records the interpreter, installed versions, repository revision, and tested device in a relative output path. Training compatibility remains a separate check for the training step.

In [5]:
import json
from datetime import datetime, timezone

record = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "python_architecture": platform.machine(),
    "os": platform.platform(),
    "repository_commit": repo_commit,
    "device_tested": str(DEVICE),
    "cuda_available": cuda_available,
    "mps_available": mps_available,
    "packages": {name: metadata.version(name) for name in selected},
    "package_imports_passed": True,
    "torch_forward_backward_passed": True,
    "cub_training_verified": False,
}
output_dir = PROJECT_DIR / "setup_logs"
output_dir.mkdir(exist_ok=True)
(output_dir / "environment.json").write_text(json.dumps(record, indent=2) + "\n")
print("Saved setup_logs/environment.json")
print("Environment checks passed. A CUB training run has not been verified yet.")

Saved setup_logs/environment.json
Environment checks passed. A CUB training run has not been verified yet.


## 3. Data Download & Preparation

### 3.1 Manual download and source

The user **manually downloaded and extracted** CUB-200-2011 from the [official Caltech dataset page](https://www.vision.caltech.edu/datasets/cub_200_2011/) linked in the technical assessment, then placed the extracted data in `CUB-200-2011/` in this project. This notebook has not downloaded the dataset.

The background audit verified all **11,788 source JPGs** exist, are nonempty, and pass Pillow header/`verify()` checks. It also checked complete image IDs, class labels, splits, bounding boxes, image–attribute pairs, and image–part pairs. Header verification is not a full pixel decode or a training-pipeline check.

### 3.2 Directory structure and loader root

The extracted data has an extra nested `CUB_200_2011/` directory. `DATA_DIR`, configured near the top of the notebook, must be its **outer** directory. The repository appends `CUB_200_2011/` itself.

```text
CUB-200-2011/
├── CUB_200_2011/                     ← DATA_DIR / --data_dir
│   ├── attributes.txt               ← downloaded attribute definitions
│   └── CUB_200_2011/                 ← dataset directory
│       ├── attributes.txt           ← copy placed here for the loader
│       ├── README
│       ├── images/                  ← 200 species folders, 11,788 JPGs
│       ├── cropped/                 ← 11,788 generated bounding-box crops
│       ├── images.txt
│       ├── image_class_labels.txt
│       ├── train_test_split.txt
│       ├── bounding_boxes.txt
│       ├── classes.txt
│       ├── attributes/
│       │   └── image_attribute_labels.txt
│       └── parts/
│           └── part_locs.txt
└── segmentations/                   ← 11,788 PNG masks; unused by this loader
```

**Placement fix:** the cell below copies the existing outer `attributes.txt` into the inner dataset directory, preserving the downloaded original. No additional dataset download is needed. Before this fix, the repository's integrity check failed because it could not find the attribute definitions.

For a clean clone, `scripts/download_cub.py` downloads the official archive and recreates this loader-compatible layout. It verifies the published archive MD5 and places `attributes.txt`. Our dataset remains the manually downloaded copy documented above. Segmentation masks are optional and omitted by the script. Dataset files are excluded from Git.

### Evidence for the attribute-file location

The following excerpts are quoted from our checked-out repository revision recorded in Section 2. `_load_attributes(root)` explicitly reads the definitions from `root/CUB_200_2011/attributes.txt`:

[cub2011parts.py, lines 178–179](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L178-L179)

```python
        attr_list = pd.read_csv(os.path.join(root, 'CUB_200_2011', 'attributes.txt'),
                                sep=' ', names=['attr_id', 'def'])
```

The repository's own downloader also relocates the file from the outer root to that same inner directory:

[cub2011parts.py, lines 280–283](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L280-L283)

```python
        # Move `attributes.txt` to basefolder instead of root
        if os.path.exists(os.path.join(self.root, 'attributes.txt')):
            os.rename(os.path.join(self.root, 'attributes.txt'),
                      os.path.join(self.root, 'CUB_200_2011', 'attributes.txt'))
```

Here, `root` is our `DATA_DIR`. We use a copy rather than the downloader's rename so the manually downloaded original remains available. The next cell checks byte equality and then runs the original dataset's integrity/metadata initialization with **cropping and downloading disabled**. This checks the layout without preparing image tensors or running training.

In [6]:
import hashlib
import shutil
from data.cub2011parts import CUB2011Parts_dataset

DATASET_DIR = DATA_DIR / "CUB_200_2011"
ATTRIBUTE_SOURCE = DATA_DIR / "attributes.txt"
ATTRIBUTE_DESTINATION = DATASET_DIR / "attributes.txt"
assert ATTRIBUTE_SOURCE.is_file(), f"Missing source: {ATTRIBUTE_SOURCE}"
if not ATTRIBUTE_DESTINATION.exists():
    shutil.copy2(ATTRIBUTE_SOURCE, ATTRIBUTE_DESTINATION)
    print("Copied attribute definitions to:", ATTRIBUTE_DESTINATION)
else:
    print("Attribute definitions already present:", ATTRIBUTE_DESTINATION)
assert ATTRIBUTE_DESTINATION.read_bytes() == ATTRIBUTE_SOURCE.read_bytes(), "Attribute files differ."
print("Source and destination SHA-256:", hashlib.sha256(ATTRIBUTE_DESTINATION.read_bytes()).hexdigest())

# Validate metadata and image paths only; do not fetch samples with this uncropped dataset.
metadata_train = CUB2011Parts_dataset(root=str(DATA_DIR), train=True, crop=False, download=False)
assert len(metadata_train) == 5994 and metadata_train.num_classes == 200
print("Original loader integrity/metadata check passed: 5,994 training images, 200 classes.")

Attribute definitions already present: /Users/mengyuanz/Documents/projects/concept_transformer_explain/CUB-200-2011/CUB_200_2011/CUB_200_2011/attributes.txt
Source and destination SHA-256: d5b158c197a8548071114150b2384c55cbd8881b32e57816c6ccc1437e946a1a


Original loader integrity/metadata check passed: 5,994 training images, 200 classes.


In [7]:
import pandas as pd

DATASET_DIR = DATA_DIR / "CUB_200_2011"
ATTRIBUTE_SOURCE = DATA_DIR / "attributes.txt"
ATTRIBUTE_DESTINATION = DATASET_DIR / "attributes.txt"
SEGMENTATION_DIR = PROJECT_DIR / "CUB-200-2011" / "segmentations"

assert DATASET_DIR.is_dir(), f"Check DATA_DIR: {DATA_DIR}"
print("Loader --data_dir:", DATA_DIR)
print("Dataset directory:", DATASET_DIR)
print("Top-level dataset contents:")
for entry in sorted(DATASET_DIR.iterdir()):
    if not entry.name.startswith("."):
        print(f"  {entry.name}{'/' if entry.is_dir() else ''}")

required = ["images.txt", "image_class_labels.txt", "train_test_split.txt",
            "bounding_boxes.txt", "classes.txt", "attributes/image_attribute_labels.txt",
            "parts/part_locs.txt"]
assert all((DATASET_DIR / name).is_file() for name in required)
images = pd.read_csv(DATASET_DIR / "images.txt", sep=" ", names=["image_id", "path"])
splits = pd.read_csv(DATASET_DIR / "train_test_split.txt", sep=" ", names=["image_id", "train"])
assert len(images) == 11788 and images.image_id.is_unique
assert len(splits) == 11788 and splits.image_id.is_unique
assert set(images.image_id) == set(splits.image_id)
assert set(splits.train) == {0, 1}
missing_images = [name for name in images.path if not (DATASET_DIR / "images" / name).is_file()]
assert not missing_images, f"Missing images: {missing_images[:5]}"

def count_rows(path):
    with path.open() as handle:
        return sum(bool(line.strip()) for line in handle)

attribute_definitions = ATTRIBUTE_DESTINATION if ATTRIBUTE_DESTINATION.is_file() else ATTRIBUTE_SOURCE
assert attribute_definitions.is_file(), "Attribute definitions are missing."
summary = {
    "source images": len(images),
    "classes": count_rows(DATASET_DIR / "classes.txt"),
    "official training images": int(splits.train.sum()),
    "official test images": int((splits.train == 0).sum()),
    "attribute definitions": count_rows(attribute_definitions),
    "image–attribute rows": count_rows(DATASET_DIR / "attributes/image_attribute_labels.txt"),
    "image–part rows": count_rows(DATASET_DIR / "parts/part_locs.txt"),
    "segmentation masks (unused)": sum(1 for _ in SEGMENTATION_DIR.rglob("*.png")),
}
assert summary["classes"] == 200 and summary["attribute definitions"] == 312
assert summary["official training images"] == 5994
assert summary["official test images"] == 5794
assert summary["image–attribute rows"] == 11788 * 312
assert summary["image–part rows"] == 11788 * 15
for label, value in summary.items():
    print(f"{label}: {value:,}")
print("Attribute definitions in loader location:", ATTRIBUTE_DESTINATION.is_file())
CROP_DIR = DATASET_DIR / "cropped"
missing_crops = [name for name in images.path if not (CROP_DIR / name).is_file()]
print(f"Bounding-box crops present: {len(images) - len(missing_crops):,}/{len(images):,}")
print("Ready for default cropped-image loader:", ATTRIBUTE_DESTINATION.is_file() and not missing_crops)

Loader --data_dir: /Users/mengyuanz/Documents/projects/concept_transformer_explain/CUB-200-2011/CUB_200_2011
Dataset directory: /Users/mengyuanz/Documents/projects/concept_transformer_explain/CUB-200-2011/CUB_200_2011/CUB_200_2011
Top-level dataset contents:
  README
  attributes/
  attributes.txt
  bounding_boxes.txt
  classes.txt
  cropped/
  image_class_labels.txt
  images/
  images.txt
  parts/
  train_test_split.txt


source images: 11,788
classes: 200
official training images: 5,994
official test images: 5,794
attribute definitions: 312
image–attribute rows: 3,677,856
image–part rows: 176,820
segmentation masks (unused): 11,788
Attribute definitions in loader location: True
Bounding-box crops present: 11,788/11,788
Ready for default cropped-image loader: True


### 3.3 Preprocessing required before training

| Operation | Repository behavior | Current status |
| --- | --- | --- |
| Place attribute definitions | Expects `DATA_DIR/CUB_200_2011/attributes.txt`. | Copied into the expected location and verified by the original loader. |
| Crop to bird bounding boxes | `CUB2011Parts_dataset(crop=True)` crops all source images and saves JPEGs under `CUB_200_2011/cropped/`. | Complete: 11,788 crops generated with the original repository code. |
| Resize and normalize | Dataset transforms resize to 224 × 224 and use ImageNet mean/std. | Applied when loading batches; no separate offline resize needed. |
| Training augmentation | Horizontal flip (probability 0.5) and rotation within ±30°; keypoints transform with the image. | Applied during training only. |
| Concept targets | Loader selects 108 concepts from 312 attributes: 13 global and 95 spatial; part locations produce targets on a 14 × 14 patch grid. | Generated by the loader; no external concept-mapping file needed. |

The attribute-file placement fix and bounding-box cropping are complete. The cell below uses `crop=True, download=False` to reproduce preparation from the manually downloaded data, or reuse the complete crop cache on subsequent runs.

The repository only checks whether the crop directory exists before skipping cropping. If a crop run is interrupted, an incomplete cache must be handled before retrying; the cell checks for that condition instead of silently using partial data.

The source attribute file has 606 rows with surplus trailing fields. The installed repository parser retains the image ID, attribute ID, and presence values used by the model; the unused annotation-time field can be truncated on those rows. No annotation rewrite is needed for the current loader.

### Evidence for generating the bounding-box crops

The repository README documents running `python cvit_cub.py`; it does not provide a precomputed crop download. The official Caltech download page provides original images/annotations and segmentations, without listing this repository's `cropped/` cache. For this workflow, **we generate the crops locally using the supplied repository code**.

The dataset constructor defaults to `crop=True`:

[cub2011parts.py, lines 61–62](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L61-L62)

```python
    def __init__(self, root='~/data/cub2011', train=True, crop=True,
                 transform=None, target_transform=None, download=False, resize_size=None):
```

It creates the crop cache and calls `_crop_images()` when that directory is absent:

[cub2011parts.py, lines 75–80](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L75-L80)

```python
        # Process images by cropping them
        if self.crop:
            if not os.path.exists(os.path.join(self.root, self.cropped_folder)):
                os.mkdir(os.path.join(self.root, self.cropped_folder))
                self._crop_images()
            self.sample_folder = self.cropped_folder
```

The implementation loads each source image, crops with the bounding-box coordinates, and saves the result at the corresponding path inside `cropped/`:

[cub2011parts.py, lines 285–297](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L285-L297)

```python
    def _crop_images(self):
        """Process images by cropping them and saving them in `cropped_folder`
        """
        for img_id, sample in tqdm(self.data.iterrows(), total=len(self.data),
                                   desc="Cropping and saving images..."):
            path = os.path.join(self.root, self.base_folder, sample.filepath)
            img = self.loader(path)

            proc_img = img.crop(sample[['x1', 'y1', 'x2', 'y2']])
            proc_filename = os.path.join(self.root, self.cropped_folder, sample.filepath)
            if not os.path.exists(os.path.dirname(proc_filename)):
                os.mkdir(os.path.dirname(proc_filename))
            proc_img.save(proc_filename)
```

The crop happens before the constructor splits training and test images, so the first run prepares all 11,788 images. In `data/cub2011parts_datamodule.py`, `prepare_data()` invokes this constructor with the comment `# Download and crop`; its default enables cropping. We can invoke the same constructor with `download=False` because our data is already on disk.

Sources: [repository README](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/README.md#L76-L84), [data-module preparation](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L34-L36), and [official Caltech downloads](https://www.vision.caltech.edu/datasets/cub_200_2011/).

### Completed crop run

The background subagent ran the unchanged repository constructor with `crop=True, download=False`. Metadata loading and crop generation took **46.00 seconds**; generation and verification took **49.14 seconds after imports**. The cache contains **11,788 JPEGs in 200 species directories**, occupying **132.90 MiB**.

**Verification passed:** all **11,788 crops** were nonempty and passed Pillow `verify()` checks; 200 representative crops also fully decoded. A transformed loader sample passed its checks: the repository's test transform produced an image of shape `(3, 224, 224)`, global concepts `(13,)`, spatial concepts `(196, 95)`, and a valid zero-indexed class label. These checks establish data-pipeline readiness; they do not establish model or training correctness.

Audit: [setup_logs/cub-cropping.json](setup_logs/cub-cropping.json). Log: [setup_logs/cub-cropping.log](setup_logs/cub-cropping.log).

In [8]:
missing_crops = [name for name in images.path if not (CROP_DIR / name).is_file()]
if CROP_DIR.exists() and missing_crops:
    raise RuntimeError("Crop directory exists but is incomplete. Inspect it before retrying preprocessing.")
cache_existed = CROP_DIR.exists()
prepared_train = CUB2011Parts_dataset(root=str(DATA_DIR), train=True, crop=True, download=False)
assert len(prepared_train) == 5994
crop_paths = [CROP_DIR / name for name in images.path]
assert all(path.is_file() and path.stat().st_size > 0 for path in crop_paths)
print("Reused complete crop cache." if cache_existed else "Generated crop cache with the original repository code.")
print(f"Verified {len(crop_paths):,} referenced, nonempty crops.")
print(f"Crop cache size: {sum(path.stat().st_size for path in crop_paths) / 1024**2:.2f} MiB")
print("Attribute placement and bounding-box crops are ready for the next step.")

Reused complete crop cache.
Verified 11,788 referenced, nonempty crops.
Crop cache size: 132.90 MiB
Attribute placement and bounding-box crops are ready for the next step.


### Loader-managed preprocessing and augmentation

After bounding-box crops are generated, the repository's data module and dataset handle the remaining preprocessing and augmentation when samples are fetched. We do not need to save a separate resized, normalized, or augmented dataset.

#### 1. Image resizing

Both training and evaluation pipelines resize each crop to **224 × 224**:

[data/cub2011parts_datamodule.py, lines 22–22](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L22-L22)

```python
        self.train_transform = A.Compose([Resize(224, 224),
```

[data/cub2011parts_datamodule.py, lines 29–29](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L29-L29)

```python
        self.test_transform = A.Compose([Resize(224, 224),
```

#### 2. Training augmentation

Training applies a horizontal flip with probability **0.5** and samples a rotation within **±30°** on every call (`p=1.0`). These operations happen during sample loading, so augmentation can vary across visits to the same image:

[data/cub2011parts_datamodule.py, lines 23–24](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L23-L24)

```python
                                          HorizontalFlip(p=0.5),
                                          Rotate(limit=(-30,30),p=1.0),
```

Validation and test use the evaluation pipeline, which has no random flip or rotation:

[data/cub2011parts_datamodule.py, lines 29–32](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L29-L32)

```python
        self.test_transform = A.Compose([Resize(224, 224),
                                         Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
                                         ToTensorV2()],
                                        keypoint_params = A.KeypointParams(format='xy', remove_invisible=False))
```

#### 3. Image normalization

Both pipelines normalize the image with ImageNet channel means `[0.485, 0.456, 0.406]` and standard deviations `[0.229, 0.224, 0.225]`:

[data/cub2011parts_datamodule.py, lines 25–25](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L25-L25)

```python
                                          Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
```

[data/cub2011parts_datamodule.py, lines 30–30](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L30-L30)

```python
                                         Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
```

#### 4. Conversion to a PyTorch tensor

Both pipelines use `ToTensorV2()` to convert the image into the tensor consumed by the model. After resizing and conversion, its shape is `(3, 224, 224)`:

[data/cub2011parts_datamodule.py, lines 26–26](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L26-L26)

```python
                                          ToTensorV2()],
```

[data/cub2011parts_datamodule.py, lines 31–31](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L31-L31)

```python
                                         ToTensorV2()],
```

#### 5. Alignment of part coordinates with the transformed image

The dataset first subtracts the bounding-box origin from spatial part coordinates so they are relative to the cropped image:

[data/cub2011parts.py, lines 326–327](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L326-L327)

```python
            else:
                part_coordinates.append((row.x - bbox_x, row.y - bbox_y))
```

Both pipelines configure `xy` keypoints and keep them in the returned keypoint list even if they move outside the image:

[data/cub2011parts_datamodule.py, lines 27–27](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L27-L27)

```python
                                         keypoint_params = A.KeypointParams(format='xy', remove_invisible=False))
```

[data/cub2011parts_datamodule.py, lines 32–32](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L32-L32)

```python
                                        keypoint_params = A.KeypointParams(format='xy', remove_invisible=False))
```

`__getitem__` passes the image and keypoints through the same transform, then stores the transformed coordinates. This keeps the spatial annotations aligned with resizing, flipping, and rotation:

[data/cub2011parts.py, lines 331–341](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L331-L341)

```python
        if self.transform is not None:
            image_np = np.array(img)
            transformed = self.transform(image=image_np, keypoints=part_coordinates)
            img = transformed['image']
            kps = transformed['keypoints']

            kps_idx = 0
            for i, row in current_attrs_parts.iterrows():
                current_attrs_parts.at[i, 'x_t'] = int(kps[kps_idx][0])
                current_attrs_parts.at[i, 'y_t'] = int(kps[kps_idx][1])
                kps_idx += 1
```

#### 6. Class labels and concept targets

The dataset shifts class IDs from 1–200 to 0–199:

[data/cub2011parts.py, lines 312–312](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L312-L312)

```python
        target = sample_metadata.target - 1  # Targets start at 1 by default, so shift to 0
```

It then constructs global and patch-level concept targets using the transformed annotations and returns the complete sample. Global targets have shape `(13,)`; spatial targets have shape `(196, 95)`:

[data/cub2011parts.py, lines 346–348](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts.py#L346-L348)

```python
        expl, spatial_expl = self._get_explanations(idx, current_attrs_parts, img.shape)

        return (img, expl, spatial_expl, target)
```

#### 7. Connecting the preprocessing pipelines to the datasets

`CUB2011Parts` defines the pipelines and passes them to `CUB2011Parts_dataset`. A standalone dataset must also be given the appropriate transform. Training uses `train_transform`; the test dataset, including the validation subset drawn from it, uses `test_transform`:

[data/cub2011parts_datamodule.py, lines 38–40](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/data/cub2011parts_datamodule.py#L38-L40)

```python
    def setup(self, stage=None):
        trainset = CUB2011Parts_dataset(train=True, root=self.data_dir, transform=self.train_transform)
        testset = CUB2011Parts_dataset(train=False, root=self.data_dir, transform=self.test_transform)
```

### 3.4 Split and model-download considerations

The source data has 5,994 training images and 5,794 test images. The original data module samples **1,000 validation images from the official test split**, then uses the **entire official test split** for final testing. Validation and final test therefore overlap; this must be disclosed when interpreting reproduced metrics. We have not changed that behavior.

The CUB download is complete. The training script separately requests pretrained `vit_large_patch16_224` weights through `timm`. The debug run in Section 4 downloaded these weights into the project-local `.cache/torch/hub/checkpoints/` directory, configured through `TORCH_HOME`. Subsequent runs reuse that cache. A regular batch-size-32 training run is now in progress; full results are pending.

## 4. Training Run

### 4.1 Small debug run

We ran the **repository model and training pipeline**, with a one-line debug-only test fix documented below, using `--debug`, **batch size 2**, and **0 data-loader workers**, on the Python 3.8 Rosetta **CPU** environment from Section 2. The runner in `scripts/run_debug.py` invokes the repository CLI, captures stdout/stderr, and records the exact command, working directory, repository revision, and elapsed time.

The repository maps `--debug` to Lightning `fast_dev_run=True`: one training batch (including backward propagation and an optimizer step), one validation batch, and one test batch. It still instantiates the full ViT-Large model, approximately 310 million trainable parameters. This checks whether the pipeline executes; it is not a full training run or a reproduction of the paper's accuracy.

```bash
uv run --locked python scripts/run_debug.py
```

The wrapper passes:

```text
cvit_cub.py --data_dir <DATA_DIR> --debug --batch_size 2 --num_workers 0
```

The working directory is `training_runs/debug/`. Pretrained weights are cached under `.cache/torch/`, set through `TORCH_HOME`; the first attempt downloaded an approximately 1.13 GiB `.npz` checkpoint. Model defaults (including learning rate 0.00005, explanation-loss weight 1.0, and attention sparsity 0.5) are preserved. `fast_dev_run` limits execution regardless of the CLI's default epoch setting.

### 4.2 Issues encountered and resolutions

#### Batch-size-1 loss error

Our initial batch-size-1 run reached the training loss and failed with:

```text
IndexError: The shape of the mask [196] at index 0 does not match
  the shape of the indexed tensor [1, 196, 95] at index 0
```

The original spatial concept loss uses an unrestricted `.squeeze()`:

[ctc/ctc.py, lines 613–617](https://github.com/IBM/concept_transformer/blob/84c9c30108a6249e017d580c4120ea54af0064f7/ctc/ctc.py#L613-L617)

```python
    norm = attn_targets.sum(-1, keepdims=True)
    idx = ~torch.isnan(norm).squeeze()
    if not torch.any(idx):
        return 0.0
    norm_attn_targets = (attn_targets[idx] / norm[idx]).float()
```

For batch size 1, it removes the batch dimension along with the trailing singleton dimension, producing a mask `(196,)` for targets `(1, 196, 95)`. We increased batch size to **2** so the mask retains shape `(2, 196)`, preserving the original repository code. The failed attempt is retained under `training_runs/debug/batch-size-1/`.

#### Debug evaluation expects a checkpoint

The batch-size-2 retry completed training and validation but the original final call `trainer.test()` raised:

```text
MisconfigurationException: You cannot execute `.test()` with
`fast_dev_run=True` unless you do `.test(ckpt_path=PATH)` as no checkpoint
path was generated during fitting.
```

Debug mode disables checkpoint saving. We made a **one-line change in `cvit_cub.py`** so debug testing uses the trained model already in memory, while normal runs retain the original `trainer.test()` behavior:

```python
test_results = trainer.test(model=model, ckpt_path=None) if args.debug else trainer.test()
```

The model, losses, optimizer, and normal training/evaluation path are unchanged. The source diff is saved in `training_runs/debug/code.diff`, and the modified entry-point SHA-256 is recorded in `run.json`. The failed retry is retained under `training_runs/debug/batch-size-2-checkpoint-error/`.

### 4.3 Execute or inspect the recorded run

On the current project, the next cell displays the successful recorded run. Set `RERUN_DEBUG = True` to rerun it; if there is no run record, the cell executes it automatically. The first run in a clean environment needs network access for the pretrained weights. Raw output is saved in `training_runs/debug/run.log`, and run metadata in `training_runs/debug/run.json`.

In [9]:
import json
import shlex
import subprocess
import re
import ast

RERUN_DEBUG = False
DEBUG_DIR = PROJECT_DIR / "training_runs" / "debug"
DEBUG_RECORD = DEBUG_DIR / "run.json"
if RERUN_DEBUG or not DEBUG_RECORD.exists():
    subprocess.run([sys.executable, str(PROJECT_DIR / "scripts" / "run_debug.py")],
                   cwd=str(PROJECT_DIR), check=True)

debug_record = json.loads(DEBUG_RECORD.read_text())
assert debug_record["success"] and debug_record["returncode"] == 0, "Inspect training_runs/debug/run.log."
assert debug_record["repository_commit"] == repo_commit, "Recorded run used a different repository revision."
assert debug_record["entrypoint_sha256"] == hashlib.sha256((REPO_DIR / "cvit_cub.py").read_bytes()).hexdigest(), "Entry point changed; rerun debug."
print("Exact command:", shlex.join(debug_record["command"]))
print("Working directory:", debug_record["working_directory"])
print("Mode:", debug_record["mode"])
print("Exit code:", debug_record["returncode"])
print(f"Elapsed time: {debug_record['elapsed_seconds']:.2f} seconds (includes model loading)")

debug_log = (DEBUG_DIR / "run.log").read_text()
print("\nOne-batch test results (pipeline check only):")
match = re.search(r"DATALOADER:0 TEST RESULTS\s*(\{.*?\})", debug_log, flags=re.DOTALL)
assert match, "Missing test-results dictionary."
debug_metrics = ast.literal_eval(match.group(1))
for name in ["test_acc", "test_ce_loss", "test_expl_loss", "test_l1_loss"]:
    value = float(debug_metrics[name])
    assert np.isfinite(value), f"Non-finite test metric: {name}"
    print(f"{name}: {value:.6f}")
print("\nRaw output:", DEBUG_DIR / "run.log")

Exact command: /Users/mengyuanz/Documents/projects/concept_transformer_explain/.venv/bin/python /Users/mengyuanz/Documents/projects/concept_transformer_explain/concept_transformer/cvit_cub.py --data_dir /Users/mengyuanz/Documents/projects/concept_transformer_explain/CUB-200-2011/CUB_200_2011 --debug --batch_size 2 --num_workers 0
Working directory: /Users/mengyuanz/Documents/projects/concept_transformer_explain/training_runs/debug
Mode: repository --debug (fast_dev_run), with debug-only in-memory test fix
Exit code: 0
Elapsed time: 49.82 seconds (includes model loading)

One-batch test results (pipeline check only):
test_acc: 0.000000
test_ce_loss: 5.283626
test_expl_loss: 1.352253
test_l1_loss: 0.245224

Raw output: /Users/mengyuanz/Documents/projects/concept_transformer_explain/training_runs/debug/run.log


### 4.4 What this run establishes

The debug CLI completed training, validation, and testing on CPU with the original model and loss functions. Its recorded one-batch test results above are **not estimates of dataset accuracy**: only two test images are evaluated after a single optimizer update.

Lightning disables normal logging and checkpoint callbacks in `fast_dev_run` mode. We captured the console output separately; this run does not provide a meaningful loss curve or a trained checkpoint. The warning about having few data-loader workers is expected because we deliberately use `num_workers=0` for this small run. The regular training run is documented next; its full results are pending.

See [debug-only source diff](training_runs/debug/code.diff), [successful run log](training_runs/debug/run.log), [run metadata](training_runs/debug/run.json), and [initial batch-size-1 failure](training_runs/debug/batch-size-1/run.log), and [debug checkpoint failure](training_runs/debug/batch-size-2-checkpoint-error/run.log).

### 4.5 Regular training started with batch size 32

At the user's request, regular training was started with **batch size 32**, matching `cvit_cub.py`'s default, **50 epochs**, and **0 data-loader workers**, on the Rosetta CPU environment. An initial regular run with batch size 2 was deliberately interrupted to make this change; its logs are preserved.

`scripts/run_training.py` invokes the repository entry point with the original model, loss, optimizer, learning-rate schedule, logging, and checkpoint behavior. A timing-only Lightning callback records batch durations and training loss. This run is ongoing: the snapshot below does not establish that an epoch or the full experiment has finished. The PID in the run record belongs to the original machine.

```bash
uv run --locked python scripts/run_training.py \
  --run-dir training_runs/my-run --batch-size 32 --max-epochs 50
```

Regular training is not automatically restarted by running this notebook. Unlike the debug run, it produces normal TensorBoard logs and checkpoints. Dataset files, pretrained weights, and generated checkpoints are excluded from Git. We need several completed training batches to refine the full-run time estimate.

In [10]:
import json
from statistics import median

ACTIVE_RUN = PROJECT_DIR / "training_runs" / "active_run.json"
if ACTIVE_RUN.exists():
    active = json.loads(ACTIVE_RUN.read_text())
    regular_dir = PROJECT_DIR / active["run_dir"]
    regular_record = json.loads((regular_dir / "run.json").read_text())
    print("Recorded regular run:", active["run_dir"])
    print("Recorded status (not a live-process check):", regular_record["status"])
    print("Batch size:", regular_record["batch_size"])
    print("Configured epochs:", regular_record["max_epochs"])
    timing_path = regular_dir / "batch_timings.jsonl"
    rows = []
    if timing_path.exists():
        for line in timing_path.read_text().splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass  # A live writer may not have finished the final line.
    train_rows = [row for row in rows if row["phase"] == "train"]
    print("Completed training batches recorded:", len(train_rows))
    if len(train_rows) >= 5:
        seconds = median(row["wall_seconds"] for row in train_rows[2:])
        batches = regular_record["train_batches_per_epoch"]
        print(f"Median batch time after the first two batches: {seconds:.2f} seconds")
        print(f"Training-only estimate per epoch: {seconds * batches / 3600:.2f} hours")
        print("Validation, checkpoint writing, and final testing add time.")
    else:
        print("Not enough consecutive training batches for a reliable new estimate yet.")
else:
    print("No regular training snapshot is available.")

Recorded regular run: training_runs/regular-b32-20261009-232419
Recorded status (not a live-process check): training
Batch size: 32
Configured epochs: 50
Completed training batches recorded: 0
Not enough consecutive training batches for a reliable new estimate yet.
